# Interactive Sampling Explorer

Inspect saved interactive sampling runs stored under `sampling_experiments/artifacts`.
Use the widgets below to pick a run, select a graph, and scrub through expansion steps while
visualizing node positions, active leaves, and expansion probabilities.

## Usage
1. Run `python sampling_experiments/runners/run_interactive_sampling.py ...` to generate artifacts.
2. Refresh the run dropdown (re-run the cell) if you add new runs.
3. Select a run, graph, and step to visualize predictions and metadata.
4. Scroll below the plot to inspect per-leaf logits/probabilities for the chosen step.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'sampling_experiments').is_dir():
            return candidate
    raise RuntimeError('Could not locate repo root containing sampling_experiments/')

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"[notebook] Using repo root: {REPO_ROOT}")

In [ ]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sampling_experiments.interactive_methods import (
    GraphStepTrace,
    AugmentedGraphStepTrace,
)

In [ ]:
ARTIFACT_ROOT = (REPO_ROOT / 'sampling_experiments' / 'artifacts')
RUN_CACHE: dict[Path, dict] = {}
TRACE_CACHE: dict[tuple[Path, int], list] = {}


def available_runs():
    if not ARTIFACT_ROOT.exists():
        return []
    runs = [p for p in ARTIFACT_ROOT.iterdir() if p.is_dir()]
    return sorted(runs, key=lambda p: p.stat().st_mtime, reverse=True)


def load_run_summary(run_dir: Path) -> dict:
    run_dir = run_dir.resolve()
    if run_dir not in RUN_CACHE:
        summary_path = run_dir / "run_summary.json"
        if not summary_path.exists():
            raise FileNotFoundError(f"run_summary.json missing in {run_dir}")
        with open(summary_path, "r") as f:
            RUN_CACHE[run_dir] = json.load(f)
    return RUN_CACHE[run_dir]


def load_graph_trace(run_dir: Path, graph_idx: int):
    key = (run_dir.resolve(), graph_idx)
    if key not in TRACE_CACHE:
        trace_path = run_dir / f"graph_{graph_idx}_trace.pkl"
        if not trace_path.exists():
            raise FileNotFoundError(f"Trace file missing: {trace_path}")
        with open(trace_path, "rb") as f:
            TRACE_CACHE[key] = pickle.load(f)
    return TRACE_CACHE[key]


def _to_numpy(tensor):
    if tensor is None:
        return None
    if hasattr(tensor, "detach"):
        return tensor.detach().cpu().numpy()
    return np.asarray(tensor)


def plot_graph_step(step_trace: GraphStepTrace):
    pos = _to_numpy(step_trace.node_positions)
    if pos is None or pos.shape[0] == 0:
        raise ValueError("Step trace has no node positions")
    fig, ax = plt.subplots(figsize=(5, 5))
    edges = step_trace.edges or []
    for u, v in edges:
        pts = pos[[u, v], :2]
        ax.plot(pts[:, 0], pts[:, 1], color="lightgray", linewidth=1, zorder=1)
    num_nodes = pos.shape[0]
    node_colors = np.full(num_nodes, "#4c72b0")
    if step_trace.leaf_ids:
        node_colors = np.array(["#4c72b0"] * num_nodes)
        for idx in step_trace.leaf_ids:
            if 0 <= idx < num_nodes:
                node_colors[idx] = "#dd8452"
    ax.scatter(pos[:, 0], pos[:, 1], c=node_colors, s=40, edgecolors="k", linewidths=0.4, zorder=2)
    ax.set_title(f"Step {step_trace.step_idx} | nodes={num_nodes} | leaves={len(step_trace.leaf_ids)}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    fig.tight_layout()
    return fig


def leaf_dataframe(step_trace: GraphStepTrace) -> pd.DataFrame:
    leaf_ids = step_trace.leaf_ids or []
    rows = []
    probs = step_trace.expansion_probs or []
    logits = step_trace.expansion_logits or []
    global_ids = []
    if step_trace.extras and step_trace.extras.get("leaf_global_ids"):
        global_ids = step_trace.extras["leaf_global_ids"]
    for idx, leaf in enumerate(leaf_ids):
        rows.append(
            {
                "leaf_local": leaf,
                "leaf_global": global_ids[idx] if idx < len(global_ids) else None,
                "logit": logits[idx] if idx < len(logits) else None,
                "prob": probs[idx] if idx < len(probs) else None,
            }
        )
    return pd.DataFrame(rows)


def describe_step(step_trace: GraphStepTrace) -> str:
    remaining = step_trace.remaining_capacity
    expanded = None
    if step_trace.extras and step_trace.extras.get("expanded_leaf_ids"):
        expanded = step_trace.extras["expanded_leaf_ids"]
    parts = [f"Remaining capacity: {remaining}"]
    if expanded:
        parts.append(f"Expanded leaves (local ids): {expanded}")
    return " | ".join(parts)

In [ ]:
run_dropdown = widgets.Dropdown(description="Run:")
graph_dropdown = widgets.Dropdown(description="Graph:")
step_slider = widgets.IntSlider(description="Step:", min=0, max=0, value=0)
output = widgets.Output()


def refresh_runs():
    runs = available_runs()
    options = [(p.name, p.name) for p in runs]
    if not options:
        run_dropdown.options = [("<no runs>", None)]
        run_dropdown.value = None
        graph_dropdown.options = []
        step_slider.max = 0
        return
    run_dropdown.options = options
    run_dropdown.value = options[0][1]


def update_graph_options():
    run_name = run_dropdown.value
    if run_name is None:
        graph_dropdown.options = []
        step_slider.max = 0
        return
    run_dir = ARTIFACT_ROOT / run_name
    summary = load_run_summary(run_dir)
    graph_meta = summary.get("graphs", [])
    if not graph_meta:
        graph_dropdown.options = []
        step_slider.max = 0
        return
    graph_options = [
        (f"#{g['graph_idx']} | nodes={g['num_nodes']} | steps={g['num_traces']}", g["graph_idx"] )
        for g in graph_meta
    ]
    graph_dropdown.options = graph_options
    graph_dropdown.value = graph_options[0][1]
    step_slider.max = graph_meta[0]["num_traces"] - 1
    step_slider.value = 0


def update_step_slider():
    run_name = run_dropdown.value
    graph_idx = graph_dropdown.value
    if run_name is None or graph_idx is None:
        step_slider.max = 0
        return
    run_dir = ARTIFACT_ROOT / run_name
    traces = load_graph_trace(run_dir, graph_idx)
    step_slider.max = max(len(traces) - 1, 0)
    if step_slider.value > step_slider.max:
        step_slider.value = step_slider.max


def render_current_step(*args):
    output.clear_output()
    run_name = run_dropdown.value
    graph_idx = graph_dropdown.value
    if run_name is None or graph_idx is None:
        with output:
            display(Markdown("**No run selected.**"))
        return
    run_dir = ARTIFACT_ROOT / run_name
    traces = load_graph_trace(run_dir, graph_idx)
    if not traces:
        with output:
            display(Markdown("**Trace is empty.**"))
        return
    step_idx = min(step_slider.value, len(traces) - 1)
    trace = traces[step_idx]
    fig = plot_graph_step(trace)
    df = leaf_dataframe(trace)
    desc = describe_step(trace)
    with output:
        display(Markdown(f"### Run `{run_name}` — Graph {graph_idx} — Step {step_idx}"))
        display(Markdown(desc))
        display(fig)
        plt.close(fig)
        if not df.empty:
            display(df)
        else:
            display(Markdown("_No active leaves at this step._"))


run_dropdown.observe(lambda change: (update_graph_options(), update_step_slider(), render_current_step()), names="value")
graph_dropdown.observe(lambda change: (update_step_slider(), render_current_step()), names="value")
step_slider.observe(lambda change: render_current_step(), names="value")

refresh_runs()
update_graph_options()
update_step_slider()
render_current_step()

widgets.VBox([
    widgets.HBox([run_dropdown, graph_dropdown]),
    step_slider,
    output,
])